[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_07/notebook_7_4_ecg_classification_1d_cnn.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 7.4: ECG Signal Classification for AFib Detection

**Chapter 7: Time Series Analysis - Implementing Journey 2 (Yuki)**

**Journey Connection**: This notebook implements Journey 2 from Chapter 3, where Yuki received multiple false AFib alerts from a wearable device. For the clinical context and patient story, refer to Chapter 3.

## Learning Objectives

1. Preprocess ECG waveforms (filtering, quality assessment)
2. Extract hand-crafted features (R-R intervals, HRV)
3. Train a rhythm classifier two ways: a Random Forest on HRV features (runs everywhere, including the browser) and a 1D CNN on the raw ECG (needs TensorFlow), and compare accuracy, interpretability and computational cost
4. Analyze false positives and motion artifacts, and add a signal-quality gate that rejects unreadable strips
5. Understand the impact of prevalence on PPV

## Clinical Context

Atrial fibrillation increases stroke risk 5×. Wearable ECG devices can catch paroxysmal AFib missed in office visits.

**Yuki's story** (Chapter 3, Journey 2): Yuki Tanaka, 55, a healthy marketing executive who exercises regularly, bought an ECG smartwatch after her mother's stroke from undiagnosed AFib. During a 6:12 AM jog it alerted: *"Possible atrial fibrillation."* A 12-lead ECG and a 48-hour Holter were normal: probably a false alarm from motion artifact. The alerts kept coming, and she eventually stopped wearing the watch continuously.

**The challenge**: In a 55-year-old without known heart disease, AFib prevalence is only ~0.1–0.5%. At 0.5% (Chapter 3), a device with 95% sensitivity and 98% specificity has a PPV of only ~19%: about **4 false alarms for every true AFib detection**.

---

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.stats import kurtosis, skew
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Generate Synthetic ECG Data

We'll create **synthetic** single-lead ECG waveforms for normal sinus rhythm and atrial fibrillation. Both rhythms use the same QRS-T template, so they differ by **rhythm** (regular vs irregularly irregular R-R intervals), plus P waves (normal) vs small fibrillatory f-waves (AFib).

To make the task realistically hard, the generator includes mechanisms that confuse real devices:
- **Subtle AFib**: some episodes are rate-controlled with only mildly irregular R-R intervals (SD from 0.06 s)
- **Paroxysmal AFib**: in ~30% of AFib recordings the episode starts late in the 30 s window, after normal beats
- **Ectopic beats**: a few normal recordings contain premature beats with compensatory pauses
- Variable sensor noise, heart rate (55–95 bpm sinus, 75–115 bpm AFib) and sinus arrhythmia

By default there is **no motion artifact**: Chapter 3 notes that the 95% sensitivity / 98% specificity reported for wearable AFib detectors comes from clinical validation data recorded at rest. Section 5 adds the motion of Yuki's morning jog. The knobs were set so that, averaged over several random seeds, the classifiers land near those published figures.

In [2]:
def ecg_beat_template(fs=250, p_wave=True):
    """One PQRST complex (0.6 s); the R peak sits 0.20 s after the beat start."""
    tb = np.arange(int(0.6 * fs)) / fs
    beat = (1.0 * np.exp(-((tb - 0.200) / 0.012) ** 2)      # R
            - 0.10 * np.exp(-((tb - 0.180) / 0.010) ** 2)   # Q
            - 0.25 * np.exp(-((tb - 0.225) / 0.010) ** 2)   # S
            + 0.30 * np.exp(-((tb - 0.400) / 0.050) ** 2))  # T
    if p_wave:
        beat += 0.12 * np.exp(-((tb - 0.080) / 0.025) ** 2)  # P
    return beat


def add_motion_artifact(ecg, intensity=0.5, fs=250, segment=None):
    """Add a motion artifact (low-frequency baseline swings from arm/body movement)."""
    artifact = intensity * np.random.randn(len(ecg))
    artifact = signal.filtfilt(*signal.butter(2, 6.0, fs=fs), artifact)  # < 6 Hz
    artifact *= intensity / (artifact.std() + 1e-12)
    if segment is not None:                      # only part of the recording
        mask = np.zeros(len(ecg)); mask[segment[0]:segment[1]] = 1
        artifact *= mask
    return ecg + artifact


def generate_synthetic_ecg(duration=30, fs=250, rhythm='normal',
                           ectopic_prob=0.02, ectopic_burden=(0.02, 0.06), motion_prob=0.0, motion_intensity=(0.15, 0.45),
                           af_rr_sd=(0.06, 0.20), paroxysmal_prob=0.3, paroxysmal_onset=(0.7, 0.95),
                           noise_range=(0.03, 0.10)):
    """
    Generate a SYNTHETIC single-lead wearable ECG.

    Parameters:
    - duration: seconds
    - fs: sampling frequency (Hz)
    - rhythm: 'normal' or 'afib'
    - ectopic_prob: chance a NORMAL recording contains premature (ectopic) beats
    - ectopic_burden: fraction of beats that are premature in those recordings
    - motion_prob: chance a recording (either rhythm) has a motion-artifact segment (exercise);
      0 by default, like clinical validation data recorded at rest (Section 5 adds motion)
    - motion_intensity: artifact amplitude range (mV)
    - af_rr_sd: range of R-R standard deviation in AFib (s); low values = rate-controlled,
      subtle AFib that is hard to tell from sinus rhythm with ectopics
    - paroxysmal_prob: chance an AFib recording is PAROXYSMAL (starts as sinus rhythm,
      AFib begins at a random point given by paroxysmal_onset, as a fraction of the recording)
    - noise_range: range of sensor noise SD (mV)

    Both rhythms share the same QRS-T template, so the classes differ by rhythm
    (and P waves vs. f-waves), not by QRS amplitude.
    """
    t = np.arange(0, duration, 1/fs)
    n_beats = int(duration / 0.3) + 2

    # Sinus R-R intervals (used for normal recordings, and before a paroxysmal AFib episode)
    hr = np.random.uniform(55, 95)
    rr_sinus = np.random.normal(60 / hr, np.random.uniform(0.01, 0.04), n_beats)
    if rhythm == 'normal' and np.random.rand() < ectopic_prob:
        # Premature beats: early by 20-40%, followed by a compensatory pause
        p_ect = np.random.uniform(*ectopic_burden)
        for i in range(n_beats - 1):
            if np.random.rand() < p_ect:
                early = rr_sinus[i] * np.random.uniform(0.2, 0.4)
                rr_sinus[i] -= early
                rr_sinus[i + 1] += early

    af_start = np.inf  # time (s) at which AFib begins
    if rhythm == 'afib':
        # AFib: irregularly irregular R-R intervals, no P waves
        hr_mean = np.random.uniform(75, 115)
        rr_sd = np.random.uniform(*af_rr_sd)
        rr_af = np.random.normal(60 / hr_mean, rr_sd, n_beats)
        too_short = rr_af < 0.3                 # truncate at 0.3 s (200 bpm):
        while too_short.any():                  # redraw the impossible ones
            rr_af[too_short] = np.random.normal(60 / hr_mean, rr_sd, too_short.sum())
            too_short = rr_af < 0.3
        # Paroxysmal AFib: the episode starts part-way through the recording
        af_start = duration * np.random.uniform(*paroxysmal_onset) if np.random.rand() < paroxysmal_prob else 0.0

    beat_sinus = ecg_beat_template(fs, p_wave=True)
    beat_af = ecg_beat_template(fs, p_wave=False)
    ecg = np.zeros(len(t) + len(beat_sinus))
    beat_time = np.random.uniform(0, 1.0)  # random phase
    for i in range(n_beats):
        idx = int(beat_time * fs)
        if idx >= len(t):
            break
        in_af = beat_time >= af_start
        ecg[idx:idx + len(beat_sinus)] += beat_af if in_af else beat_sinus
        beat_time += rr_af[i] if in_af else rr_sinus[i]
    ecg = ecg[:len(t)]

    if rhythm == 'afib':
        # Fibrillatory f-waves: small 4-8 Hz baseline oscillation (often barely visible)
        f = np.random.uniform(4, 8)
        f_wave = np.random.uniform(0.01, 0.05) * np.sin(2 * np.pi * f * t + np.random.uniform(0, 2 * np.pi))
        ecg += f_wave * (t >= af_start)

    # Sensor noise (varies with skin contact)
    ecg += np.random.normal(0, np.random.uniform(*noise_range), len(ecg))

    # Motion artifact during part of the recording (e.g., a morning jog)
    if np.random.rand() < motion_prob:
        seg_len = int(np.random.uniform(0.2, 0.6) * len(t))
        start = np.random.randint(0, len(t) - seg_len)
        ecg = add_motion_artifact(ecg, np.random.uniform(*motion_intensity), fs, (start, start + seg_len))

    return t, ecg

# Generate examples
t_normal, ecg_normal = generate_synthetic_ecg(duration=10, rhythm='normal')
t_afib, ecg_afib = generate_synthetic_ecg(duration=10, rhythm='afib')

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(t_normal, ecg_normal, 'b-', linewidth=1)
axes[0].set_title('Normal Sinus Rhythm (synthetic)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Amplitude (mV)')
axes[0].set_xlim([0, 10])
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_afib, ecg_afib, 'r-', linewidth=1)
axes[1].set_title('Atrial Fibrillation (synthetic)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Amplitude (mV)')
axes[1].set_xlim([0, 10])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Peak amplitude: normal {ecg_normal.max():.2f} mV vs AFib {ecg_afib.max():.2f} mV (same QRS template)")
print("Key differences:")
print("  Normal: Regular R-R intervals, visible P waves")
print("  AFib: Irregular R-R intervals, absent P waves, small fibrillatory baseline waves")


Peak amplitude: normal 1.12 mV vs AFib 1.13 mV (same QRS template)
Key differences:
  Normal: Regular R-R intervals, visible P waves
  AFib: Irregular R-R intervals, absent P waves, small fibrillatory baseline waves


## 2. Feature-Based Classification

Traditional approach: Extract hand-crafted features from ECG signal.

In [3]:
def extract_rr_intervals(ecg, fs=250):
    """Detect R-peaks and calculate R-R intervals."""
    # Simple peak detection: half the height of the tallest R waves,
    # refractory period 0.25 s (allows heart rates up to 240 bpm)
    threshold = 0.5 * np.percentile(ecg, 99.5)
    peaks, _ = signal.find_peaks(ecg, height=threshold, distance=int(fs * 0.25))

    rr_intervals = np.diff(peaks) / fs  # Convert to seconds
    return peaks, rr_intervals

def extract_hrv_features(rr_intervals):
    """Calculate heart rate variability features."""
    if len(rr_intervals) < 2:
        return {}

    features = {
        'rr_mean': np.mean(rr_intervals),
        'rr_std': np.std(rr_intervals),
        'rr_rmssd': np.sqrt(np.mean(np.diff(rr_intervals)**2)),
        'rr_min': np.min(rr_intervals),
        'rr_max': np.max(rr_intervals),
        'rr_range': np.max(rr_intervals) - np.min(rr_intervals),
        'hr_mean': 60 / np.mean(rr_intervals),
        'hr_std': np.std(60 / rr_intervals),
    }
    return features

# Extract features from examples
peaks_normal, rr_normal = extract_rr_intervals(ecg_normal)
peaks_afib, rr_afib = extract_rr_intervals(ecg_afib)

features_normal = extract_hrv_features(rr_normal)
features_afib = extract_hrv_features(rr_afib)

print("Normal Sinus Rhythm Features:")
for k, v in features_normal.items():
    print(f"  {k}: {v:.3f}")

print("\nAtrial Fibrillation Features:")
for k, v in features_afib.items():
    print(f"  {k}: {v:.3f}")

if features_afib['rr_std'] > features_normal['rr_std']:
    print(f"\n💡 Notice: AFib has higher variability (rr_std {features_afib['rr_std']:.2f} vs {features_normal['rr_std']:.2f} s)")
else:
    print("\n💡 Notice: AFib does NOT show higher rr_std in this example")
if features_normal['rr_std'] > 0.1:
    print(f"   Caution: the 'normal' signal also looks variable (detected HR {features_normal['hr_mean']:.0f} bpm):")
    print("   the simple threshold peak detector misses or adds beats. Check peaks before trusting HRV features.")

Normal Sinus Rhythm Features:
  rr_mean: 0.870
  rr_std: 0.029
  rr_rmssd: 0.044
  rr_min: 0.840
  rr_max: 0.928
  rr_range: 0.088
  hr_mean: 68.980
  hr_std: 2.280

Atrial Fibrillation Features:
  rr_mean: 0.701
  rr_std: 0.009
  rr_rmssd: 0.018
  rr_min: 0.684
  rr_max: 0.716
  rr_range: 0.032
  hr_mean: 85.592
  hr_std: 1.152

💡 Notice: AFib does NOT show higher rr_std in this example


## 3. Generate Dataset and Train Classifier

Create a larger dataset and train Random Forest on HRV features. We keep the raw waveforms too, for the 1D CNN in Section 3b, and use the **same** train/test split for both models.

In [4]:
# Generate dataset
n_samples = 1000
X_features = []
y_labels = []
X_raw = []  # raw waveforms for the 1D CNN (Section 3b)

print("Generating dataset...")
for i in range(n_samples):
    rhythm = 'normal' if i < n_samples // 2 else 'afib'
    label = 0 if rhythm == 'normal' else 1

    _, ecg = generate_synthetic_ecg(duration=30, rhythm=rhythm)
    peaks, rr = extract_rr_intervals(ecg)
    features = extract_hrv_features(rr)

    if len(features) > 0:
        X_features.append(list(features.values()))
        y_labels.append(label)
        X_raw.append(ecg)

X = np.array(X_features)
y = np.array(y_labels)
X_raw = np.array(X_raw)

feature_names = list(features_normal.keys())

print(f"Dataset shape: {X.shape}")
print(f"Features: {feature_names}")
print(f"Class distribution: {np.bincount(y)}")

# Split data (split indices, so the 1D CNN can reuse exactly the same split)
idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.3, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = X[idx_train], X[idx_test], y[idx_train], y[idx_test]

# Train Random Forest
import time
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
t0 = time.perf_counter()
rf.fit(X_train, y_train)
rf_train_time = time.perf_counter() - t0

# Evaluate
y_pred = rf.predict(X_test)
y_pred_proba = rf.predict_proba(X_test)[:, 1]

print("\nPerformance on Test Set:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'AFib']))

auroc = roc_auc_score(y_test, y_pred_proba)
print(f"AUROC: {auroc:.3f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'AFib'], yticklabels=['Normal', 'AFib'])
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix: Feature-Based Classification')
plt.show()

Generating dataset...


Dataset shape: (1000, 8)
Features: ['rr_mean', 'rr_std', 'rr_rmssd', 'rr_min', 'rr_max', 'rr_range', 'hr_mean', 'hr_std']
Class distribution: [500 500]

Performance on Test Set:
              precision    recall  f1-score   support

      Normal       0.96      0.99      0.98       150
        AFib       0.99      0.96      0.98       150

    accuracy                           0.98       300
   macro avg       0.98      0.98      0.98       300
weighted avg       0.98      0.98      0.98       300

AUROC: 0.991


## 3b. 1D CNN on the Raw ECG

The book (Section 7.5) describes a 1D CNN with convolutional layers of increasing width (**32 → 64 → 128 filters**), each followed by ReLU and max pooling, then **global average pooling** and dense layers. We train exactly that on the raw waveforms, using the same train/test split as the Random Forest, and compare:

- **Accuracy** (test accuracy, AUROC)
- **Interpretability** (feature importances vs. a black box)
- **Computational cost** (parameters, training time, inference time per 30 s ECG)

Design note: the input is downsampled to 125 Hz and the kernels are 15 samples wide, so the last layer "sees" about 2.4 s of signal: enough to compare 2–3 consecutive beats, which is what rhythm classification needs.

This section needs **TensorFlow** (Colab or local Jupyter). In the browser (JupyterLite) it is skipped and the feature-based classifier above is the result.

In [5]:
try:
    import tensorflow as tf
    HAS_TF = True
except ImportError:
    HAS_TF = False
    print("ℹ️ This section needs TensorFlow (Colab or local Jupyter); it is skipped in the browser version.")

if HAS_TF:
    tf.random.set_seed(42)
    tf.keras.utils.set_random_seed(42)

    # Preprocess: downsample 250 -> 125 Hz, z-score each recording
    X_cnn = X_raw[:, ::2]
    X_cnn = (X_cnn - X_cnn.mean(axis=1, keepdims=True)) / (X_cnn.std(axis=1, keepdims=True) + 1e-8)
    X_cnn = X_cnn[..., np.newaxis].astype('float32')
    Xc_train, Xc_test = X_cnn[idx_train], X_cnn[idx_test]

    L = tf.keras.layers
    cnn = tf.keras.Sequential([
        L.Input(shape=X_cnn.shape[1:]),
        L.Conv1D(32, 15, padding='same', activation='relu'),  L.MaxPooling1D(4),
        L.Conv1D(64, 15, padding='same', activation='relu'),  L.MaxPooling1D(4),
        L.Conv1D(128, 15, padding='same', activation='relu'),
        L.GlobalAveragePooling1D(),
        L.Dense(32, activation='relu'),
        L.Dense(1, activation='sigmoid'),
    ])
    cnn.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])

    t0 = time.perf_counter()
    history = cnn.fit(Xc_train, y_train, epochs=15, batch_size=32, validation_split=0.15, verbose=0)
    cnn_train_time = time.perf_counter() - t0

    cnn_proba = cnn.predict(Xc_test, verbose=0).ravel()
    cnn_pred = (cnn_proba >= 0.5).astype(int)
    print(f"1D CNN: {cnn.count_params():,} parameters, trained {len(history.history['loss'])} epochs in {cnn_train_time:.1f} s")
    print(f"Final training accuracy {history.history['accuracy'][-1]:.3f}, validation accuracy {history.history['val_accuracy'][-1]:.3f}")
    print("\nPerformance on Test Set:")
    print(classification_report(y_test, cnn_pred, target_names=['Normal', 'AFib']))
    print(f"AUROC: {roc_auc_score(y_test, cnn_proba):.3f}")


I0000 00:00:1795590212.127865    7219 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795590212.128453    7219 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795590213.559520    7219 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795590213.559929    7219 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


1D CNN: 158,465 parameters, trained 15 epochs in 11.4 s
Final training accuracy 0.975, validation accuracy 0.952

Performance on Test Set:
              precision    recall  f1-score   support

      Normal       0.88      1.00      0.94       150
        AFib       1.00      0.87      0.93       150

    accuracy                           0.93       300
   macro avg       0.94      0.93      0.93       300
weighted avg       0.94      0.93      0.93       300

AUROC: 0.991


### Feature-Based vs. 1D CNN: Accuracy, Interpretability, Cost

Inference time is measured per single 30 s recording (as a wearable would process it), median of 50 runs. For the Random Forest it **includes** R-peak detection and HRV feature extraction; for the CNN it includes only the forward pass (preprocessing is negligible). The CNN's "size" is its trainable parameters; the forest's is its number of tree nodes (each node stores a feature index and threshold).

In [6]:
from sklearn.metrics import accuracy_score

def median_ms(fn, n=50):
    times = []
    for i in range(n):
        t0 = time.perf_counter(); fn(i); times.append((time.perf_counter() - t0) * 1000)
    return np.median(times)

rf_acc = accuracy_score(y_test, y_pred)
rf_auc = roc_auc_score(y_test, y_pred_proba)
rf_nodes = sum(est.tree_.node_count for est in rf.estimators_)
rf_ms = median_ms(lambda i: rf.predict_proba(
    [list(extract_hrv_features(extract_rr_intervals(X_raw[idx_test[i % len(idx_test)]])[1]).values())]))

def sens_spec(y_true, y_hat):
    y_true, y_hat = np.asarray(y_true), np.asarray(y_hat)
    return (y_hat[y_true == 1] == 1).mean(), (y_hat[y_true == 0] == 0).mean()

rows = [('HRV features + Random Forest', rf_acc, rf_auc, *sens_spec(y_test, y_pred), f'{rf_nodes:,} tree nodes', rf_train_time, rf_ms)]
if HAS_TF:
    cnn_acc = accuracy_score(y_test, cnn_pred)
    cnn_auc = roc_auc_score(y_test, cnn_proba)
    cnn_ms = median_ms(lambda i: cnn(Xc_test[i % len(Xc_test)][np.newaxis], training=False))
    rows.append(('1D CNN (raw ECG)', cnn_acc, cnn_auc, *sens_spec(y_test, cnn_pred), f'{cnn.count_params():,} parameters', cnn_train_time, cnn_ms))

print(f"{'Model':<30} {'Accuracy':>9} {'AUROC':>7} {'Sens':>6} {'Spec':>6} {'Size':>22} {'Train (s)':>10} {'Infer (ms/ECG)':>15}")
print("-" * 111)
for name, acc, auc_, se, sp, size, tt, ms in rows:
    print(f"{name:<30} {acc:>9.3f} {auc_:>7.3f} {se:>6.1%} {sp:>6.1%} {size:>22} {tt:>10.2f} {ms:>15.2f}")
print(f"{'Chapter 3 wearable (clinical validation)':<40} {'':>6} {'95.0%':>6} {'98.0%':>6}")

# Interpretability: the forest tells us which features drove its decisions
imp = sorted(zip(feature_names, rf.feature_importances_), key=lambda x: -x[1])[:3]
print("\n🔍 Interpretability: top Random Forest features: " + ", ".join(f"{n} ({v:.2f})" for n, v in imp))
print("   The CNN has no built-in equivalent; it would need saliency methods (Chapter 10).")

if HAS_TF:
    diff = cnn_acc - rf_acc
    if abs(diff) < 0.01:
        verdict = "about as accurate as"
    elif diff > 0:
        verdict = f"{diff:.1%} MORE accurate than"
    else:
        verdict = f"{-diff:.1%} LESS accurate than"
    print(f"\n💡 On this synthetic data the 1D CNN is {verdict} the HRV-feature Random Forest,")
    print(f"   took {cnn_train_time / max(rf_train_time, 1e-9):.0f}× longer to train, and is "
          f"{cnn_ms / rf_ms:.1f}× {'slower' if cnn_ms > rf_ms else 'faster'} per ECG at inference.")
    if diff <= 0.01:
        print("   When the signal is rhythm irregularity, hand-crafted HRV features already capture it;")
        print("   the CNN's advantage appears with subtler morphology and much more training data.")
else:
    print("\nℹ️ CNN comparison skipped (no TensorFlow); the feature-based classifier is the browser result.")


Model                           Accuracy   AUROC   Sens   Spec                   Size  Train (s)  Infer (ms/ECG)
---------------------------------------------------------------------------------------------------------------
HRV features + Random Forest       0.977   0.991  96.0%  99.3%       3,952 tree nodes       0.07            3.50
1D CNN (raw ECG)                   0.933   0.991  86.7% 100.0%     158,465 parameters      11.36            3.69
Chapter 3 wearable (clinical validation)         95.0%  98.0%

🔍 Interpretability: top Random Forest features: rr_range (0.31), rr_std (0.26), hr_std (0.17)
   The CNN has no built-in equivalent; it would need saliency methods (Chapter 10).

💡 On this synthetic data the 1D CNN is 4.3% LESS accurate than the HRV-feature Random Forest,
   took 167× longer to train, and is 1.1× slower per ECG at inference.
   When the signal is rhythm irregularity, hand-crafted HRV features already capture it;
   the CNN's advantage appears with subtler morpholog

## 4. The Prevalence Problem

Even with 98% specificity, screening low-risk populations yields poor PPV.

In [7]:
def calculate_ppv(sensitivity, specificity, prevalence):
    """Calculate positive predictive value."""
    tp = sensitivity * prevalence
    fp = (1 - specificity) * (1 - prevalence)
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    fp_ratio = fp / tp if tp > 0 else float('inf')
    return ppv, fp_ratio

# From confusion matrix
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print(f"Random Forest performance (test set, recorded at rest like clinical validation data):")
print(f"  Sensitivity: {sensitivity:.1%}   (Chapter 3 device: 95%)")
print(f"  Specificity: {specificity:.1%}   (Chapter 3 device: 98%)")
ppv_ours, fpr_ours = calculate_ppv(sensitivity, specificity, 0.005)
print(f"  → at Yuki's 0.5% prevalence: PPV {ppv_ours:.0%}, {fpr_ours:.1f} false alarms per true AFib (Chapter 3: ≈19%, ≈4)")
print(f"  (only {tn + fp} normal test ECGs: each false positive moves specificity by {1/(tn + fp):.1%})")

# Calculate PPV for different settings
settings = [
    ('Hospital (symptomatic)', 0.10),
    ('Yuki: 55 y, low-risk (0.5%)', 0.005),   # Chapter 3 base-rate example
    ('Low-risk, lower bound (0.1%)', 0.001)    # Chapter 3 range: 0.1-0.5%
]

print("\n" + "="*70)
print("POSITIVE PREDICTIVE VALUE BY SETTING")
print("="*70)
for setting, prev in settings:
    ppv, fp_ratio = calculate_ppv(sensitivity, specificity, prev)
    print(f"{setting:30s} | Prevalence: {prev:6.1%} | PPV: {ppv:6.1%} | FP ratio: {fp_ratio:5.1f}:1")
print("="*70)

if specificity == 1.0:
    print(f"\n⚠️  Our model made no false positives on {tn + fp} normal test ECGs, so PPV looks perfect.")
    print("   That reflects easy synthetic data and a small test set, not a real-world 0% false-alarm rate.")

# Illustrative wearable with realistic, imperfect performance
ill_sens, ill_spec = 0.95, 0.98
print(f"\nIllustrative device (sensitivity {ill_sens:.0%}, specificity {ill_spec:.0%}):")
for setting, prev in settings:
    ppv, fp_ratio = calculate_ppv(ill_sens, ill_spec, prev)
    print(f"{setting:30s} | Prevalence: {prev:6.1%} | PPV: {ppv:6.1%} | FP ratio: {fp_ratio:5.1f}:1")

ppv_yuki, fp_ratio_yuki = calculate_ppv(ill_sens, ill_spec, 0.005)
n_pop = 10_000
print(f"\nChapter 3 arithmetic for {n_pop:,} people like Yuki (prevalence 0.5%):")
print(f"   {n_pop*0.005:.0f} have AFib → {n_pop*0.005*ill_sens:.1f} detected; "
      f"{n_pop*0.995:,.0f} healthy → {n_pop*0.995*(1-ill_spec):.0f} false alarms")
print("\n⚠️  CLINICAL INSIGHT:")
print("   Same model, different settings → vastly different clinical utility")
print(f"   For Yuki (prevalence 0.5%), {ill_sens:.0%} sensitivity and {ill_spec:.0%} specificity → PPV {ppv_yuki:.0%}, "
      f"≈{fp_ratio_yuki:.0f} false alarms per true AFib ({1-ppv_yuki:.0%} of alerts are false)")

Random Forest performance (test set, recorded at rest like clinical validation data):
  Sensitivity: 96.0%   (Chapter 3 device: 95%)
  Specificity: 99.3%   (Chapter 3 device: 98%)
  → at Yuki's 0.5% prevalence: PPV 42%, 1.4 false alarms per true AFib (Chapter 3: ≈19%, ≈4)
  (only 150 normal test ECGs: each false positive moves specificity by 0.7%)

POSITIVE PREDICTIVE VALUE BY SETTING
Hospital (symptomatic)         | Prevalence:  10.0% | PPV:  94.1% | FP ratio:   0.1:1
Yuki: 55 y, low-risk (0.5%)    | Prevalence:   0.5% | PPV:  42.0% | FP ratio:   1.4:1
Low-risk, lower bound (0.1%)   | Prevalence:   0.1% | PPV:  12.6% | FP ratio:   6.9:1

Illustrative device (sensitivity 95%, specificity 98%):
Hospital (symptomatic)         | Prevalence:  10.0% | PPV:  84.1% | FP ratio:   0.2:1
Yuki: 55 y, low-risk (0.5%)    | Prevalence:   0.5% | PPV:  19.3% | FP ratio:   4.2:1
Low-risk, lower bound (0.1%)   | Prevalence:   0.1% | PPV:   4.5% | FP ratio:  21.0:1

Chapter 3 arithmetic for 10,000 people

## 5. Motion Artifacts

Wearable devices are especially susceptible to motion artifacts during exercise. Our classifier was trained and tested on recordings made at rest. What happens when people wear it on a morning jog, like Yuki?

To answer that we need **both** kinds of recording under motion: healthy ones (does **specificity** drop?) and AFib ones (does **sensitivity** change?). The PPV during exercise is then computed from the sensitivity and specificity measured *during exercise*, not borrowed from the resting test set. Recordings the pipeline cannot read are reported separately instead of silently counting as "no alert".

In [8]:
# Example: the same kind of normal ECG, at rest and during a jog
t_motion, ecg_clean = generate_synthetic_ecg(duration=10, rhythm='normal', ectopic_prob=0)
ecg_motion = add_motion_artifact(ecg_clean, intensity=0.3)

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(t_motion, ecg_clean, 'b-', linewidth=1)
axes[0].set_title('Clean ECG (Normal Sinus Rhythm, synthetic)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_motion, ecg_motion, 'orange', linewidth=1)
axes[1].set_title('ECG with Motion Artifact (e.g., during exercise)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t_afib, ecg_afib, 'r-', linewidth=1)
axes[2].set_title('True AFib', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Time (seconds)')
axes[2].set_ylabel('Amplitude')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Field test: 200 healthy AND 200 AFib 30 s recordings, each measured at rest and during a jog.
# During the jog, arm movement corrupts a random 10-40% of each 30 s recording.
# Signal quality: this simple pipeline has no dedicated quality gate. A recording is "unreadable"
# only if too few R peaks are found to compute HRV features. Unreadable recordings are reported
# separately and EXCLUDED from sensitivity/specificity (they are not counted as "no alert").
n_field = 200
field = {}  # (rhythm, condition) -> (alerts among evaluable recordings, number unreadable)
for rhythm in ['normal', 'afib']:
    feats = {'rest': [], 'jog': []}
    for _ in range(n_field):
        _, e = generate_synthetic_ecg(duration=30, rhythm=rhythm)
        seg = int(np.random.uniform(0.1, 0.4) * len(e)); start = np.random.randint(0, len(e) - seg)
        e_jog = add_motion_artifact(e, intensity=np.random.uniform(0.05, 0.2), segment=(start, start + seg))
        for sig, which in [(e, 'rest'), (e_jog, 'jog')]:
            f = extract_hrv_features(extract_rr_intervals(sig)[1])
            if f:
                feats[which].append(list(f.values()))
    for which, rows_ in feats.items():
        alerts = rf.predict(np.array(rows_)) == 1 if rows_ else np.array([], bool)
        field[rhythm, which] = (alerts, n_field - len(rows_))

def field_summary(which):
    """Sensitivity and specificity among EVALUABLE recordings, plus the share that was unreadable."""
    (a_af, u_af), (a_no, u_no) = field['afib', which], field['normal', which]
    sens = a_af.mean() if len(a_af) else float('nan')
    spec = 1 - a_no.mean() if len(a_no) else float('nan')
    return sens, spec, (u_af + u_no) / (2 * n_field)

(sens_rest, spec_rest, unread_rest), (sens_jog, spec_jog, unread_jog) = field_summary('rest'), field_summary('jog')
ppv_rest, fa_rest = calculate_ppv(sens_rest, spec_rest, 0.005)
ppv_jog, fa_jog = calculate_ppv(sens_jog, spec_jog, 0.005)

print(f"Field test on {n_field} healthy + {n_field} AFib synthetic recordings (same recordings, with and without motion):")
print(f"{'Condition':<10} {'Sensitivity':>12} {'Specificity':>12} {'Unreadable':>11} {'False alerts':>13} {'Missed AFib':>12} {'PPV @0.5%':>10} {'FA per true':>12}")
for which, se, sp, un, ppv_, fa_ in [('At rest', sens_rest, spec_rest, unread_rest, ppv_rest, fa_rest),
                                     ('Jogging', sens_jog, spec_jog, unread_jog, ppv_jog, fa_jog)]:
    key = 'rest' if which == 'At rest' else 'jog'
    a_af, a_no = field['afib', key][0], field['normal', key][0]
    print(f"{which:<10} {se:>12.1%} {sp:>12.1%} {un:>11.1%} {f'{a_no.sum()}/{len(a_no)}':>13} {f'{(~a_af).sum()}/{len(a_af)}':>12} {ppv_:>10.1%} {fa_:>12.1f}")
print("Sensitivity and specificity are computed among evaluable (readable) recordings only.")
print("PPV uses the sensitivity AND specificity measured in the same condition, at Yuki's 0.5% prevalence (Chapter 3).")

if unread_jog == 0:
    print("ℹ️  No recording was unreadable: without a signal-quality gate, every motion-corrupted ECG still gets a rhythm label.")
else:
    print(f"ℹ️  {unread_jog:.1%} of jogging recordings were unreadable (too few R peaks) and are excluded above.")
if spec_jog < spec_rest:
    print(f"⚠️  Specificity falls from {spec_rest:.1%} at rest to {spec_jog:.1%} while jogging: motion artifact distorts")
    print("   R-peak detection, so the R-R intervals of healthy people look irregular, like AFib.")
else:
    print("ℹ️  In this run motion did not increase false alarms in healthy recordings; try a higher intensity.")
if sens_jog < sens_rest - 0.02:
    print(f"⚠️  Sensitivity also falls ({sens_rest:.1%} → {sens_jog:.1%}): motion hides some true AFib episodes.")
elif sens_jog > sens_rest + 0.02:
    print(f"ℹ️  Sensitivity rises ({sens_rest:.1%} → {sens_jog:.1%}), but not because the detector got better:")
    print("   motion makes MORE recordings look irregular, AFib or not, so more of everything is flagged.")
else:
    print(f"ℹ️  Sensitivity is about the same ({sens_rest:.1%} at rest, {sens_jog:.1%} jogging): AFib already looks irregular.")
if ppv_jog < ppv_rest:
    print(f"   Net effect at 0.5% prevalence: PPV {ppv_rest:.0%} at rest vs {ppv_jog:.1%} during exercise "
          f"({fa_rest:.0f} vs {fa_jog:.0f} false alarms per true AFib).")
    print("   Performance measured at rest overstates real-world performance during daily activity.")
else:
    print(f"   Net effect at 0.5% prevalence: PPV {ppv_rest:.1%} at rest vs {ppv_jog:.1%} during exercise: no loss in this run.")

Field test on 200 healthy + 200 AFib synthetic recordings (same recordings, with and without motion):
Condition   Sensitivity  Specificity  Unreadable  False alerts  Missed AFib  PPV @0.5%  FA per true
At rest           99.0%        98.5%        0.0%         3/200        2/200      24.9%          3.0
Jogging           99.0%        70.0%        0.0%        60/200        2/200       1.6%         60.3
Sensitivity and specificity are computed among evaluable (readable) recordings only.
PPV uses the sensitivity AND specificity measured in the same condition, at Yuki's 0.5% prevalence (Chapter 3).
ℹ️  No recording was unreadable: without a signal-quality gate, every motion-corrupted ECG still gets a rhythm label.
⚠️  Specificity falls from 98.5% at rest to 70.0% while jogging: motion artifact distorts
   R-peak detection, so the R-R intervals of healthy people look irregular, like AFib.
ℹ️  Sensitivity is about the same (99.0% at rest, 99.0% jogging): AFib already looks irregular.
   Net eff

## 5b. A Signal-Quality Gate: "Unreadable, Please Re-record"

Section 5 ended with an uncomfortable fact: no recording was unreadable, so every motion-corrupted ECG still got a rhythm label. Real watches behave differently. Before classifying, they ask *"is this strip good enough to read?"* and, if not, answer **"inconclusive / poor recording, please try again"** instead of "possible AFib". Chapter 7 calls this **signal quality assessment**: automated quality checks keep corrupted data from producing false predictions.

**Our signal-quality index (SQI): worst-beat template correlation.** It needs only numpy/scipy, so it runs in the browser:

1. Detect R peaks (the same detector the classifier uses) and cut a ±0.10 s window around each one. The window covers only the QRS complex, which looks the same in sinus rhythm and in AFib, so the index measures *signal quality*, not *rhythm*.
2. Build the strip's own template: the median of all its windows.
3. Correlate every window with the template. A real beat correlates highly; a "beat" that is really a motion bump, or a QRS distorted by a baseline swing, does not.
4. **SQI = the lowest correlation in the strip.** One false or missed beat is enough to make R-R intervals look irregular, so the worst beat is what matters.

**Choosing the threshold.** The gate is itself a detector: it should reject strips the classifier would get wrong and keep strips it would get right. On a **validation** set we pick the threshold that maximizes

*(share of misclassified strips rejected) − (share of correctly classified strips rejected)*

and only then open the **test** set. Both sets are synthetic and mix four artifact levels (none, mild, moderate, severe), each with normal and AFib strips.

A gate does not make the classifier smarter. It converts some wrong answers (and some right ones) into *no answer*. Whether that trade is worth it is what the cell below measures.

In [9]:
def beat_template_sqi(ecg, fs=250, half_window=0.10):
    """Signal-quality index: lowest correlation between any detected beat and the strip's median beat.
    Close to 1 = every detected 'beat' looks like the others; low = at least one does not."""
    peaks, _ = extract_rr_intervals(ecg, fs)
    h = int(half_window * fs)
    peaks = peaks[(peaks >= h) & (peaks < len(ecg) - h)]
    if len(peaks) < 5:
        return -1.0                                   # too few beats to judge: unreadable
    beats = np.stack([ecg[p - h:p + h + 1] for p in peaks])
    beats = beats - beats.mean(axis=1, keepdims=True)
    template = np.median(beats, axis=0)
    corr = beats @ template / (np.linalg.norm(beats, axis=1) * np.linalg.norm(template) + 1e-12)
    return float(corr.min())

# Self-check: a train of identical beats must score ~1
assert beat_template_sqi(np.tile(ecg_beat_template(), 20)) > 0.99

# Synthetic strips with mixed artifact levels (motion amplitude in mV, on 10-40% of each 30 s strip)
ARTIFACT_LEVELS = {'none': None, 'mild': (0.03, 0.08), 'moderate': (0.08, 0.15), 'severe': (0.15, 0.30)}

def make_mixed_set(n_per_cell, seed):
    """n_per_cell strips per (artifact level, rhythm). Returns a DataFrame: level, y, pred, sqi."""
    np.random.seed(seed)
    rows, feats = [], []
    for level, amp in ARTIFACT_LEVELS.items():
        for rhythm in ['normal', 'afib']:
            for _ in range(n_per_cell):
                _, e = generate_synthetic_ecg(duration=30, rhythm=rhythm)
                if amp is not None:
                    seg = int(np.random.uniform(0.1, 0.4) * len(e)); start = np.random.randint(0, len(e) - seg)
                    e = add_motion_artifact(e, intensity=np.random.uniform(*amp), segment=(start, start + seg))
                f = extract_hrv_features(extract_rr_intervals(e)[1])
                feats.append(list(f.values()) if f else [np.nan] * len(feature_names))
                rows.append({'level': level, 'y': int(rhythm == 'afib'), 'sqi': beat_template_sqi(e)})
    df = pd.DataFrame(rows)
    feats = np.array(feats)
    ok = ~np.isnan(feats).any(axis=1)
    df['pred'] = 0                                    # no HRV features -> no alert
    df.loc[ok, 'pred'] = rf.predict(feats[ok])        # the SAME Random Forest, trained at rest (Section 3)
    df['wrong'] = df['pred'] != df['y']
    return df

val_sq = make_mixed_set(100, seed=101)    # used ONLY to choose the threshold
test_sq = make_mixed_set(100, seed=202)   # opened once, after the threshold is fixed

# ── Choose the SQI threshold on the validation set ─────────────────────────────
grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
if val_sq['wrong'].any():
    gate_score = [(val_sq.loc[val_sq['wrong'], 'sqi'] < g).mean() - (val_sq.loc[~val_sq['wrong'], 'sqi'] < g).mean()
                  for g in grid]
    sqi_threshold = float(grid[int(np.argmax(gate_score))])
else:
    gate_score, sqi_threshold = [0.0] * len(grid), float(grid[0])   # nothing to catch: most permissive gate
print(f"Validation set: {len(val_sq)} strips, {val_sq['wrong'].sum()} misclassified by the Random Forest")
print(f"Chosen SQI threshold: {sqi_threshold:.2f}  "
      f"(rejects {(val_sq.loc[val_sq['wrong'], 'sqi'] < sqi_threshold).mean():.0%} of misclassified and "
      f"{(val_sq.loc[~val_sq['wrong'], 'sqi'] < sqi_threshold).mean():.0%} of correctly classified validation strips)")

# ── Test set: without the gate vs. with the gate ───────────────────────────────
def gate_metrics(df, ref=None):
    """Sensitivity, specificity, PPV in this (50/50) set, and PPV at Yuki's 0.5% prevalence.

    The gate may accept AFib and normal strips at different rates, so the 0.5% projection
    uses alerts per ALL strips of each class in `ref` (the full test set), not the
    accepted-only sensitivity/specificity with the original prevalence."""
    ref = df if ref is None else ref
    tp = ((df['pred'] == 1) & (df['y'] == 1)).sum(); fn = ((df['pred'] == 0) & (df['y'] == 1)).sum()
    fp = ((df['pred'] == 1) & (df['y'] == 0)).sum(); tn = ((df['pred'] == 0) & (df['y'] == 0)).sum()
    se = tp / (tp + fn) if tp + fn else float('nan')
    sp = tn / (tn + fp) if tn + fp else float('nan')
    ppv_set = tp / (tp + fp) if tp + fp else float('nan')
    tp_rate = tp / (ref['y'] == 1).sum()      # true alerts per AFib strip recorded
    fp_rate = fp / (ref['y'] == 0).sum()      # false alerts per normal strip recorded
    a, b = 0.005 * tp_rate, 0.995 * fp_rate
    ppv_low = a / (a + b) if a + b else float('nan')
    fa_low = b / a if a else float('inf')
    return se, sp, ppv_set, ppv_low, fa_low, fp

test_sq['accepted'] = test_sq['sqi'] >= sqi_threshold
acc = test_sq[test_sq['accepted']]
se0, sp0, ppv0, ppvlow0, fa0, fp0 = gate_metrics(test_sq)
se1, sp1, ppv1, ppvlow1, fa1, fp1 = gate_metrics(acc, ref=test_sq)
reject_rate = 1 - test_sq['accepted'].mean()

print(f"\nTest set: {len(test_sq)} synthetic strips ({(test_sq['y'] == 1).sum()} AFib, {(test_sq['y'] == 0).sum()} normal), "
      f"{len(ARTIFACT_LEVELS)} artifact levels in equal shares")
print(f"{'':<26} {'Strips read':>12} {'Rejected':>9} {'Sensitivity':>12} {'Specificity':>12} {'False alerts':>13} {'PPV (this set)':>15} {'PPV @0.5%':>10} {'FA per true':>12}")
print(f"{'(i)  No gate':<26} {len(test_sq):>12} {0:>9.1%} {se0:>12.1%} {sp0:>12.1%} {fp0:>13} {ppv0:>15.1%} {ppvlow0:>10.1%} {fa0:>12.1f}")
print(f"{'(ii) SQI gate, accepted':<26} {len(acc):>12} {reject_rate:>9.1%} {se1:>12.1%} {sp1:>12.1%} {fp1:>13} {ppv1:>15.1%} {ppvlow1:>10.1%} {fa1:>12.1f}")
print("Rejected strips get the message \"unreadable, please re-record\" and no rhythm label.")
print("PPV (this set) holds only for this 50/50 mix; PPV @0.5% re-weights to Yuki's prevalence, counting alerts per recorded strip of each rhythm (so rejected strips count as no alert).")

# (iii) Rejection by artifact level and by rhythm
by_level = test_sq.groupby('level', sort=False).agg(
    strips=('accepted', 'size'), rejected=('accepted', lambda a: 1 - a.mean()), wrong_no_gate=('wrong', 'mean'))
by_level['wrong_after_gate'] = [test_sq[(test_sq['level'] == lv) & test_sq['accepted']]['wrong'].mean() for lv in by_level.index]
print(f"\n(iii) {'Artifact level':<15} {'Strips':>7} {'Rejected':>9} {'Misclassified, no gate':>23} {'Misclassified, accepted':>24}")
for lv, r in by_level.iterrows():
    after = f"{r['wrong_after_gate']:.1%}" if r['wrong_after_gate'] == r['wrong_after_gate'] else 'none left'
    print(f"      {lv:<15} {int(r['strips']):>7} {r['rejected']:>9.1%} {r['wrong_no_gate']:>23.1%} {after:>24}")
rej_af = 1 - test_sq.loc[test_sq['y'] == 1, 'accepted'].mean()
rej_no = 1 - test_sq.loc[test_sq['y'] == 0, 'accepted'].mean()
print(f"      Rejected by rhythm: AFib {rej_af:.1%}, normal {rej_no:.1%}")

# Figure: SQI by artifact level, with the chosen threshold
fig, ax = plt.subplots(figsize=(10, 4))
bins = np.linspace(-0.2, 1, 49)
for lv, color in zip(ARTIFACT_LEVELS, ['tab:green', 'tab:olive', 'tab:orange', 'tab:red']):
    ax.hist(test_sq.loc[test_sq['level'] == lv, 'sqi'].clip(-0.2, 1), bins=bins, alpha=0.6, color=color, label=lv)
ax.axvline(sqi_threshold, color='k', linestyle='--', label=f'threshold {sqi_threshold:.2f} (from validation)')
ax.set_xlabel('SQI: worst-beat template correlation'); ax.set_ylabel('Test strips')
ax.set_title('Signal quality by motion-artifact level (synthetic test set)', fontweight='bold')
ax.legend(title='Artifact level'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# ── Conclusions, computed from the numbers above ───────────────────────────────
print("WHAT THE GATE DID (test set):")
if len(acc) == 0:
    print("❌ The gate rejected every strip: it is useless at this threshold.")
else:
    if sp1 > sp0 + 0.01:
        print(f"✅ Specificity among accepted strips rose from {sp0:.1%} to {sp1:.1%} "
              f"({fp0} → {fp1} false AFib alerts): most motion-induced false alarms became \"please re-record\".")
        print("   " + (f"That is back at the 98% reported for wearables on clean clinical data (Chapter 3)." if sp1 >= 0.98
                       else f"That is still below the 98% reported for wearables on clean clinical data (Chapter 3)."))
    elif sp1 < sp0 - 0.01:
        print(f"❌ Specificity among accepted strips FELL ({sp0:.1%} → {sp1:.1%}): this gate hurts here.")
    else:
        print(f"ℹ️  Specificity barely changed ({sp0:.1%} → {sp1:.1%}): this gate did not remove the false alarms.")
    if abs(se1 - se0) <= 0.02:
        print(f"ℹ️  Sensitivity among accepted strips is about the same ({se0:.1%} → {se1:.1%}).")
    else:
        print(f"{'⚠️ ' if se1 < se0 else 'ℹ️ '} Sensitivity among accepted strips changed from {se0:.1%} to {se1:.1%}.")
    print(f"⚠️  The price is coverage: {reject_rate:.1%} of all strips get no answer "
          f"({by_level.loc['none', 'rejected']:.0%} of clean strips, {by_level.loc['severe', 'rejected']:.0%} of severe-artifact strips).")
    if by_level['rejected'].is_monotonic_increasing:
        print("   Rejection grows with the artifact level, which is what a quality gate should do.")
    else:
        print("   Rejection does NOT grow steadily with the artifact level: the index is not tracking artifact well.")
    if abs(rej_af - rej_no) > 0.05:
        print(f"⚠️  The gate is not rhythm-neutral: it rejects {rej_af:.1%} of AFib strips vs {rej_no:.1%} of normal ones.")
    else:
        print(f"   The gate is close to rhythm-neutral (rejects {rej_af:.1%} of AFib and {rej_no:.1%} of normal strips).")
    n_af = (test_sq['y'] == 1).sum()
    found0 = ((test_sq['pred'] == 1) & (test_sq['y'] == 1)).sum(); found1 = ((acc['pred'] == 1) & (acc['y'] == 1)).sum()
    print(f"   Of all {n_af} AFib strips, {found0} raised an alert without the gate and {found1} with it: "
          f"{found0 - found1} true AFib strips now say \"please re-record\" instead of \"possible AFib\".")
    print("   That is acceptable only if the person does re-record at rest; a gate that is never followed up hides AFib.")
    if ppvlow1 > ppvlow0:
        print(f"💡 At Yuki's 0.5% prevalence, PPV goes from {ppvlow0:.1%} to {ppvlow1:.1%} "
              f"({fa0:.0f} → {fa1:.0f} false alarms per true AFib).")
        if ppvlow1 < 0.5:
            print("   The gate removes the motion problem, not the base-rate problem of Section 4: most alerts are still false.")
        else:
            print("   In this run most alerts on accepted strips would be true even at 0.5% prevalence.")
    else:
        print(f"💡 At Yuki's 0.5% prevalence, PPV did not improve ({ppvlow0:.1%} → {ppvlow1:.1%}).")
    print("   Caveat: synthetic beats all share one shape. In real ECGs, ectopic beats with a different shape would")
    print("   lower this index too, so real devices combine several quality checks and validate them on real recordings.")

Validation set: 800 strips, 102 misclassified by the Random Forest
Chosen SQI threshold: 0.55  (rejects 87% of misclassified and 13% of correctly classified validation strips)

Test set: 800 synthetic strips (400 AFib, 400 normal), 4 artifact levels in equal shares
                            Strips read  Rejected  Sensitivity  Specificity  False alerts  PPV (this set)  PPV @0.5%  FA per true
(i)  No gate                        800      0.0%        98.0%        70.8%           117           77.0%       1.7%         59.4
(ii) SQI gate, accepted             588     26.5%        97.3%        97.6%             7           97.6%      16.8%          4.9
Rejected strips get the message "unreadable, please re-record" and no rhythm label.
PPV (this set) holds only for this 50/50 mix; PPV @0.5% applies the same sensitivity/specificity to Yuki's prevalence.

(iii) Artifact level   Strips  Rejected  Misclassified, no gate  Misclassified, accepted
      none                200      0.0%            

## 6. Key Takeaways

### What We Learned

1. **ECG rhythm classification**: Can use hand-crafted features (HRV) or deep learning (1D CNN). Section 3b printed which was more accurate and how much more compute the CNN needs on this synthetic data; with irregular R-R intervals as the main signal, simple HRV features are hard to beat

2. **Prevalence matters enormously**:
   - For the illustrative device in Section 4 (95% sensitivity, 98% specificity):
     Hospital (10% AFib): PPV ~84%; Yuki's setting (0.5% AFib): PPV ~19%, about 4 false alarms per true alert; 0.1% AFib: PPV ~4.5%
   - Compare our own classifier's sensitivity/specificity and PPV printed in Section 4: the synthetic data was calibrated to land near the published 95%/98% at rest, but with only 150 normal test ECGs a single false positive moves specificity by 0.7%
   - Same model, vastly different utility

3. **Motion artifacts**: Major challenge for wearable devices
   - Exercise, arm movement mimics irregular rhythm
   - Leads to false positives: Section 5 measured sensitivity and specificity at rest vs. during a jog on the same healthy and AFib recordings (PPV during exercise uses both), and counted unreadable recordings separately

4. **A signal-quality gate trades coverage for accuracy** (Section 5b):
   - Our index is the worst-beat template correlation; its threshold was chosen on a validation set, then applied once to a test set with mixed artifact levels
   - A gate does not make the classifier smarter: it turns some answers into "unreadable, please re-record". Section 5b printed sensitivity, specificity and PPV with and without the gate, the share of strips rejected, and the rejection rate per artifact level
   - Rejected strips include true AFib, so a gate only helps if the person actually re-records at rest
   - It addresses the motion problem, not the base-rate problem: compare the PPV at 0.5% prevalence printed with the gate against Section 4

5. **Clinical implications**:
   - Wearable AFib detection useful for high-risk individuals
   - Should not be used for mass screening of low-risk populations
   - Need confirmatory testing pathways

6. **Yuki's experience** (Journey 2): Multiple false alarms caused anxiety, unnecessary testing, and ultimately loss of trust in technology

### Connections to Book Chapters

- **Chapter 3 (Seven Journeys)**: Yuki's story provides the clinical motivation
- **Chapter 5 (Evaluation)**: Prevalence dependence of PPV, metrics in imbalanced settings
- **Chapter 7 (Time Series)**: This chapter - signal processing, frequency domain features, 1D CNNs
- **Chapter 11 (Deployment)**: Deployment considerations, user experience, regulatory pathways

### Real-World Devices

- Apple Watch, Fitbit, KardiaMobile: FDA-cleared for AFib detection
- They also return "inconclusive" or "poor recording" for strips they cannot read (the behaviour Section 5b imitates)
- Clinical guidelines: Use with confirmatory testing, not as standalone diagnostic
- Appropriate for: Patients with symptoms or known AFib for monitoring

---

## Exercises

1. **Move the quality threshold** (Section 5b, `sqi_threshold`, `grid`, `test_sq`):
   - For every value in `grid`, compute the rejection rate and `gate_metrics(test_sq[test_sq['sqi'] >= g])`, and plot specificity among accepted strips against the share of strips rejected
   - Look for: the knee of the curve, and how many strips you must throw away for each extra point of specificity
   - Then choose the threshold on `val_sq` with a different rule (e.g. the lowest threshold whose accepted strips reach 98% specificity) and repeat with other seeds in `make_mixed_set(100, seed=...)`. Is that rule as stable as the one in the notebook?

2. **Add a second quality check** (Section 5b, `beat_template_sqi`):
   - Chapter 7 also lists kurtosis and band power as quality metrics. Add one (`kurtosis` is already imported) or a baseline-swing measure (range of a 1 s moving average)
   - Reject a strip if either check fails, with both thresholds chosen on `val_sq`
   - Look for: at the same specificity among accepted strips, does the combined gate reject fewer strips? Check the per-level table (`by_level`)

3. **"Please re-record"** (Section 5b, `test_sq['accepted']`):
   - For every rejected test strip, generate a fresh strip of the same rhythm with mild or no artifact (the person sits still) and classify that instead
   - Look for: overall sensitivity, specificity and coverage after one retry, and how many AFib strips the gate had withheld are now detected. What if only half of the users re-record?

4. **How specific must a screening device be?** (Section 4, `calculate_ppv`):
   - At Yuki's 0.5% prevalence and 95% sensitivity, find the specificity needed for a PPV above 50%
   - Look for: how many normal test ECGs you would need to demonstrate that specificity, given that each false positive moves it by 1/n (Section 4 printed this for our test set)

5. **Train with motion instead of gating it** (Section 3, `generate_synthetic_ecg(duration=30, rhythm=rhythm)`):
   - Regenerate the training data with `motion_prob=0.5`, retrain the Random Forest and re-run Sections 5 and 5b
   - Look for: does the "no gate" specificity on the mixed-artifact test set improve, and is the gate still worth its rejections? With TensorFlow, repeat for the 1D CNN (Section 3b), which sees the raw waveform

6. **A feature for f-waves** (Section 2, `extract_hrv_features`; Section 3, `feature_names`):
   - All eight features describe R-R intervals. Add the 4-8 Hz band power of the ECG (`signal.welch`), where the generator puts the fibrillatory waves
   - Look for: its rank in `rf.feature_importances_`, and whether sensitivity improves on the hard cases (rate-controlled and paroxysmal AFib, Section 1)

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 7: Time Series Analysis)*  
*This implements Journey 2 (Yuki - AFib Detection) from Chapter 3*  
*For clinical context, see Chapter 3. For time series methods, see Chapter 7.*